In [3]:
!pip install -q ultralytics kagglehub

In [4]:
import glob, shutil, yaml
from pathlib import Path
import kagglehub
from ultralytics import YOLO

# Locate dataset
ROOT = kagglehub.dataset_download("sudinrupakheti/potholes")
for p in [Path(ROOT), *Path(ROOT).rglob("*")]:
    if (p / "images" / "train").exists():
        ROOT = str(p)
        break
print("ROOT:", ROOT)

OUT = "/kaggle/working/runs"

# Check class IDs
ids = set()
for f in glob.glob(f"{ROOT}/labels/*/*.txt"):
    for line in open(f):
        if line.strip():
            ids.add(line.split()[0])
print("Class IDs:", ids)
assert ids <= {"0", "1"}, f"Stray class IDs: {ids}"

# data.yaml
cfg = {
    "path": ROOT,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "pothole", 1: "not_pothole"},
}
with open("data.yaml", "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

# Train
model = YOLO("yolov8s.pt")
model.train(
    data="data.yaml",
    epochs=100,
    imgsz=960,
    batch=8,
    patience=30,
    project=OUT,
    name="pothole_v1",
    seed=0,
    plots=True,
)

# Load best
best = Path(OUT) / "pothole_v1/weights/best.pt"
model = YOLO(str(best))

# Test eval
metrics = model.val(data="data.yaml", split="test", imgsz=960)
print("pothole mAP50:", metrics.box.ap50[0])
print("pothole mAP50-95:", metrics.box.ap[0])

# Predict, pothole only
model.predict(
    source=f"{ROOT}/images/test",
    classes=[0],
    conf=0.25,
    imgsz=960,
    save=True,
    project=OUT,
    name="pred",
)

shutil.copy(best, "/kaggle/working/best.pt")
print("Done. Saved /kaggle/working/best.pt")

ROOT: /kaggle/input/datasets/sudinrupakheti/potholes
Class IDs: {'1', '0'}
Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=poth

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/100      5.06G      2.106      2.778      2.297         13        960: 100% ━━━━━━━━━━━━ 252/252 4.9it/s 51.0s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 4.9it/s 5.5s0.2s
                   all        431        764       0.56     0.0568     0.0257    0.00653

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      5.07G      2.067      2.875      2.311          9        960: 100% ━━━━━━━━━━━━ 252/252 4.7it/s 53.2s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 5.2it/s 5.2s0.2s
                   all        431        764      0.612      0.087     0.0679     0.0254

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      5.07G      2.039      2.754      2.285          9        960: 100% ━━━━━━━━━━━━ 252/252 4.8it/s 52.9s0.2ss
                 Class     Ima

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/runs/pothole_v1/weights/best.pt'

In [5]:
!find /kaggle/working -name "best.pt"

/kaggle/working/runs/pothole_v1-2/weights/best.pt
